## Analysis questions

This notebook uses the Palmer Penguins dataset to practice core statistical tests and simple linear regression.

The main questions are:

Q1. Do physical measurements differ between species?

Q2. Do male and female penguins differ in body mass?

Q3. Are flipper length and body mass related?

Q4. Is species associated with island?

Q5. Can flipper length explain variation in body mass?

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

penguins_url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"
penguins_path = DATA_RAW / "penguins.csv"

if not penguins_path.exists():
    penguins = pd.read_csv(penguins_url)
    penguins.to_csv(penguins_path, index=False)
else:
    penguins = pd.read_csv(penguins_path)
    
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [3]:
print(f"Shape: {penguins.shape}")
print("===============")
print(f"Columns: {penguins.columns.tolist()}")
print("===============")

penguins.info()

Shape: (344, 8)
Columns: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), str(3)
memory usage: 21.6 KB


In [4]:
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

In [5]:
# check percentage of missing values in each column
(penguins.isna().mean() * 100).round(2)

species              0.00
island               0.00
bill_length_mm       0.58
bill_depth_mm        0.58
flipper_length_mm    0.58
body_mass_g          0.58
sex                  3.20
year                 0.00
dtype: float64

In [6]:
# check which rows have at least one missing value
missing_rows = penguins[penguins.isna().any(axis=1)]
missing_rows

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN,2007
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN,2007
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN,2007
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN,2007
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN,2007
178,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN,2007
218,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN,2008
256,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN,2009
268,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN,2009


## Descriptive statistics

Before answering the analysis questions, I first summarise the main numeric and categorical variables. This gives a basic understanding of the dataset and helps check whether the values and group sizes look sensible.

In [13]:
measurement_cols = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

penguins[measurement_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
bill_length_mm,342.0,43.92,5.46,32.1,39.22,44.45,48.5,59.6
bill_depth_mm,342.0,17.15,1.97,13.1,15.60,17.30,18.7,21.5
flipper_length_mm,342.0,200.92,14.06,172.0,190.00,197.00,213.0,231.0
body_mass_g,342.0,4201.75,801.95,2700.0,3550.00,4050.00,4750.0,6300.0


In [14]:
categorical_cols = ["species", "island", "sex"]

for col in categorical_cols:
    print(penguins[col].value_counts(dropna=False))
    print("===============")

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64
island
Biscoe       168
Dream        124
Torgersen     52
Name: count, dtype: int64
sex
male      168
female    165
NaN        11
Name: count, dtype: int64


## Q1. Do physical measurements differ between species?

For this question, `species` is the grouping variable and the physical measurements are numeric outcome variables.

Before running a statistical test, I first compare the measurements by species to see whether the differences look meaningful in the data.

In [20]:
species_comparison = (
    penguins
    .groupby("species")[measurement_cols]
    .agg(["count", "mean", "std", "median"])
    .round(2)
    )
species_comparison

bill_length_mm                     bill_depth_mm               \
                   count   mean   std median         count   mean   std   
species                                                                   
Adelie               151  38.79  2.66  38.80           151  18.35  1.22   
Chinstrap             68  48.83  3.34  49.55            68  18.42  1.14   
Gentoo               123  47.50  3.08  47.30           123  14.98  0.98   

                 flipper_length_mm                      body_mass_g           \
          median             count    mean   std median       count     mean   
species                                                                        
Adelie     18.40               151  189.95  6.54  190.0         151  3700.66   
Chinstrap  18.45                68  195.82  7.13  196.0          68  3733.09   
Gentoo     15.00               123  217.19  6.48  216.0         123  5076.02   

                           
              std  median  
species                    
Adelie     458.57  3700.0  
Chinstrap  384.34  3700.0  
Gentoo     504.12  5000.0